# 02_core_functionsCore Functions used by `03_main`.

### Read active control entries

In [0]:
def get_active_control_entries():
    query = f"""
        SELECT s_no, zip_source, zip_source_path, zip_tgt, zip_tgt_path, unzip_tgt_path,
               is_full_load, is_incremental_load
        FROM {control_table}
        WHERE is_active = 1
    """
    return spark.sql(query).collect()

### List zip files at a source path

In [0]:
def list_zip_files(source_path):
    source_path = source_path.rstrip("/") + "/"
    return [
        f for f in dbutils.fs.ls(source_path)
        if not f.isDir() and f.name.lower().endswith(valid_extensions)
    ]

### Extract the date token from a filename 
Expected pattern: `..._<anything>_<date>.zip`, date is the second-to-last underscore-separated token, e.g. `case_data_001_20220605_20220620_20260101_235954.zip` -> `20260101`.

In [0]:
def extract_date_token(file_name):
    name_without_extension = file_name.rsplit(".", 1)[0]
    parts = name_without_extension.split("_")

    if len(parts) < min_filename_tokens:
        return None

    return parts[-2]

### Filter zip files by load type
- `is_full_load = 1`: process every zip file found at the source path (unchanged behavior).
- `is_incremental_load = 1`: only process files whose extracted date token (from `extract_date_token`) matches **today's date** (`yyyyMMdd`, cluster/job system date). Files without a matching or extractable date token are excluded.
- Files with neither flag set are excluded (nothing to process for that control row).

In [0]:
def filter_zip_files_by_load_type(zip_files, is_full_load, is_incremental_load):

    if is_full_load == 1:
        return zip_files

    if is_incremental_load == 1:
        today_token = datetime.now().strftime("%Y%m%d")
        return [
            f for f in zip_files
            if extract_date_token(f.name) == today_token
        ]

    return []

### Copy a single file and return a log record
Tags the record with `stage="COPY"` so it can share the log table with the unzip stage.

In [0]:
def copy_zip_file(file_info, target_base_path, s_no, source, file_name_extract_pattern):
    start_time = datetime.now()
    file_name = file_info.name
    target_base_path = target_base_path.rstrip("/") + "/"

    record = {
        "s_no": s_no,
        "stage": "COPY",
        "source": source,
        "file_name": file_name,
        "source_path": file_info.path,
        "target_path": None,
        "start_time": start_time,
    }

    date_token = extract_date_token(file_name)
    if date_token is None:
        record.update(status="SKIPPED", message="Could not extract date token from filename",
                       end_time=datetime.now())
        return record

    # target_folder = f"{target_base_path}/{re.search(file_name_extract_pattern, file_name).group(1)}/{date_token}/"
    target_folder = f"{target_base_path}{date_token}/"
    print("---------------target_folder",target_folder)
    target_file_path = f"{target_folder}{file_name}"
    print("---------------target_file_path",target_file_path)
    record["target_path"] = target_file_path

    try:
        dbutils.fs.mkdirs(target_folder)
        dbutils.fs.cp(file_info.path, target_file_path)
        record.update(status="SUCCESS", message="Copied successfully")
    except Exception as e:
        record.update(status="FAILED", message=str(e))

    record["end_time"] = datetime.now()
    return record

### Unzip a single copied file and return a log record
- Takes a successful COPY log record (produced by `copy_zip_file`) and extracts that exact file from its copied location (`target_path`) into the matching `yyyyMMdd` folder under `unzip_base_path`. 
- Tags the record with `stage="UNZIP"`.
- Because this works off the copy record rather than re-scanning the raw folder, a run only ever unzips the files it just copied — it will not re-extract files copied by earlier runs.

In [0]:
def unzip_zip_file(copy_record, unzip_base_path,file_name_extract_pattern):
    
    start_time = datetime.now()
    file_name = copy_record["file_name"]
    zip_path = copy_record["target_path"]
    unzip_base_path = unzip_base_path.rstrip("/") + "/"

    record = {
        "s_no": copy_record["s_no"],
        "stage": "UNZIP",
        "source": copy_record["source"],
        "file_name": file_name,
        "source_path": zip_path,
        "target_path": None,
        "start_time": start_time,
    }

    date_token = extract_date_token(file_name)

    if date_token is None:
        record.update(status="SKIPPED", message="Could not extract date token from filename",
                       end_time=datetime.now())
        return record

    target_folder = f"{unzip_base_path}{date_token}/"
    record["target_path"] = target_folder

    try:
        dbutils.fs.mkdirs(target_folder)
        local_zip_path = zip_path.replace("dbfs:", "")
        with zipfile.ZipFile(local_zip_path, "r") as zip_ref:
            zip_ref.extractall(target_folder)
        record.update(status="SUCCESS", message="Extracted successfully")
    except zipfile.BadZipFile:
        record.update(status="FAILED", message="Invalid ZIP file")
    except Exception as e:
        record.update(status="FAILED", message=str(e))

    record["end_time"] = datetime.now()
    return record

### Idempotency: check prior run status from the log table
So re-running the pipeline never redoes work that already succeeded:
- `get_latest_stage_status(stage)` looks at `log_table` and returns, for every `(s_no, file_name)`
  seen so far in that stage (`COPY` or `UNZIP`), its most recent status. A file with latest status
  `SUCCESS` is treated as done; `FAILED`/`SKIPPED` or no prior record at all means it still needs
  to be (re)processed.
- `get_pending_unzip_from_history()` finds files that were copied successfully in an **earlier**
  run but never successfully unzipped, so they get picked up for unzip even if this run doesn't
  re-copy them.

In [0]:
def get_latest_stage_status(stage):
    """
    Return {(s_no, file_name): status} with each file's most recent status
    for the given stage ('COPY' or 'UNZIP'), based on log_table.

    latest status == 'SUCCESS'  -> already done, skip this run.
    latest status != 'SUCCESS'  -> (FAILED / SKIPPED) needs reprocessing.
    no entry for the key at all -> new file, needs processing.
    """
    query = f"""
        SELECT s_no, file_name, status
        FROM (
            SELECT s_no, file_name, status,
                   ROW_NUMBER() OVER (
                       PARTITION BY s_no, file_name
                       ORDER BY end_time DESC
                   ) AS rn
            FROM {log_table}
            WHERE stage = '{stage}' AND file_name IS NOT NULL
        )
        WHERE rn = 1
    """
    rows = spark.sql(query).collect()
    return {(r["s_no"], r["file_name"]): r["status"] for r in rows}

In [0]:
def get_pending_unzip_from_history():
    """
    Files that were copied successfully in a *previous* run but have not
    yet been unzipped successfully. Returned as copy-record-shaped dicts
    (s_no, source, file_name, target_path) so they can be fed straight
    into unzip_zip_file, alongside files copied earlier in this run.
    """
    query = f"""
        WITH last_copy AS (
            SELECT s_no, source, file_name, target_path, status,
                   ROW_NUMBER() OVER (
                       PARTITION BY s_no, file_name
                       ORDER BY end_time DESC
                   ) AS rn
            FROM {log_table}
            WHERE stage = 'COPY' AND file_name IS NOT NULL
        ),
        last_unzip AS (
            SELECT s_no, file_name, status,
                   ROW_NUMBER() OVER (
                       PARTITION BY s_no, file_name
                       ORDER BY end_time DESC
                   ) AS rn
            FROM {log_table}
            WHERE stage = 'UNZIP' AND file_name IS NOT NULL
        )
        SELECT c.s_no, c.source, c.file_name, c.target_path
        FROM last_copy c
        LEFT JOIN last_unzip u
          ON c.s_no = u.s_no AND c.file_name = u.file_name AND u.rn = 1
        WHERE c.rn = 1
          AND c.status = 'SUCCESS'
          AND (u.status IS NULL OR u.status != 'SUCCESS')
    """
    rows = spark.sql(query).collect()
    return [
        {
            "s_no": r["s_no"],
            "source": r["source"],
            "file_name": r["file_name"],
            "target_path": r["target_path"],
        }
        for r in rows
    ]

## Write Logs

In [0]:
def write_logs(log_records, run_id):
    if not log_records:
        return

    rows = [
        (
            run_id,
            r.get("stage"),
            int(r["s_no"]) if r.get("s_no") is not None else None,
            r.get("source"),
            r.get("file_name"),
            r.get("source_path"),
            r.get("target_path"),
            r.get("status"),
            r.get("message"),
            r.get("start_time"),
            r.get("end_time"),
        )
        for r in log_records
    ]

    log_df = spark.createDataFrame(rows, schema=LOG_SCHEMA)
    log_df.write.format("delta").mode("append").saveAsTable(log_table)